# Notebook 3 — Cải tiến: **Safe OOF-PHE-GBX** (thay cho Notebook 2 ở Session 2)

**Kaggle:** Accelerator = **GPU T4 x2**, Internet = **On**. Thời gian ước tính: ~2,5–3,5 giờ
(33 dataset chia đôi cho 2 GPU).

### Vì sao cần v2 — chẩn đoán từ lần chạy trước
TabPFN-GBX-PHE (v1) có hạng trung bình, log-loss và accuracy tốt nhất nhưng chỉ hơn TabPFN v2
+0.0004 AUC (p = 0.11). Trọng số ensemble được chọn trên **một split validation 20 %** (thường dưới
200 dòng) nên rất nhiễu: TabPFN v2 mặc định, một trong những thành viên tốt nhất, chỉ nhận trọng số
0.01. Vấn đề nằm ở **ước lượng trọng số**, không phải ở số lượng mô hình.

### Phương pháp v2 (TabPFN v2 vẫn giữ nguyên)
1. **Dự đoán out-of-fold trên 100 % dữ liệu train** (5-fold): với TabPFN chỉ tốn thêm 5 lượt forward,
   không cần train, nên rẻ trên T4. Trọng số được chọn trên n dòng thay vì 0,2n.
2. **Dự đoán test với toàn bộ fold train làm ngữ cảnh** (TabPFN) và **bagging** 5 mô hình fold (CatBoost).
3. **Caruana ensemble selection trên OOF**, sau đó **temperature scaling** fit trên OOF (giảm log-loss).
4. Thành viên: 7 cấu hình TabPFN v2 + CatBoost + TabPFN-GBX (cross-fit **lồng nhau** để không rò rỉ).

5. **Giữ điểm mạnh của bản gốc ("do no harm")**. TabPFN v2 đã là mô hình tốt nhất trên 22/33
   dataset, nên bản chính:
   * **neo vào TabPFN v2**: việc chọn trọng số bắt đầu với 10 bản sao TabPFN v2 (co về bản gốc);
   * **cổng kiểm định**: cross-validate chính quy trình chọn ensemble trên các dòng OOF, so log-loss
     từng dòng với TabPFN v2; chỉ dùng ensemble khi cải thiện **dương và có ý nghĩa** (t-test một
     phía, z > 1.645); nếu không, **giữ nguyên dự đoán của TabPFN v2**.
   Nhờ vậy phần cải tiến được dùng ở nơi bản gốc yếu, còn nơi bản gốc mạnh thì giữ nguyên.

Một lần fit cho ra 5 biến thể để ablation: `oofphe_pfn`, `oofphe_pfn_ts`, `oofphe_all`,
`oofphe_all_ts`, `oofphe_safe` (**bản chính**).

Báo cáo có thêm kiểm định Wilcoxon cho **log-loss** và **accuracy**, mức thua tệ nhất so với bản gốc
(`Worst ΔAUC`, số dataset thua quá 0.005 AUC), và tỉ lệ cổng chọn ensemble ở nhóm dataset TabPFN v2
yếu / mạnh.

## 0. Cài đặt chung

In [ ]:
# >>>>> SỬA DÒNG NÀY: URL repo GitHub của bạn <<<<<
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/tabpfn-dl-project.git"

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "tabpfn-dl-project")
os.environ["OPENML_CACHE"] = os.path.join(WORK, "openml_cache")
print(WORK, REPO)

In [ ]:
!nvidia-smi -L
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!python tests/test_smoke.py

In [ ]:
import torch, tabpfn, subprocess, time, sys, glob, shutil, zipfile
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count(), "| tabpfn", tabpfn.__version__)
os.makedirs("logs", exist_ok=True)

def run(cmd, log):
    """Chạy lệnh shell, in log ra cell và ghi vào file."""
    print(">>", cmd)
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
        p.wait()
    if p.returncode != 0:
        print(f"!! exit code {p.returncode} (xem {log})")

def run_parallel(jobs):
    """jobs = [(cmd, gpu_id, logfile), ...] -> mỗi GPU một tiến trình, chờ tất cả."""
    procs = []
    for cmd, gpu, log in jobs:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        print(f">> [GPU {gpu}] {cmd}")
        procs.append((subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "a"),
                                       stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        for _, log in procs:
            lines = [l for l in open(log).read().splitlines() if l.strip()]
            print(f"[{log}] {lines[-1] if lines else ''}")
    print("all jobs finished")

## 1. Khôi phục kết quả trước

Để báo cáo so sánh với **mọi** phương pháp cũ (TabPFN v2-PHE, fine-tune, GBX-PHE v1, ...), upload
`final_results.zip` của lần chạy Notebook 2 thành Kaggle Dataset và gắn vào notebook.
Nếu chỉ có `session1_results.zip` thì báo cáo so với TabPFN v2 và các baseline.
Nếu không có gì, cell sẽ chạy lại Phần A (~1 giờ).

In [ ]:
BENCH = "results/benchmark"
z_final = glob.glob("/kaggle/input/**/final_results.zip", recursive=True)
z_s1 = glob.glob("/kaggle/input/**/session1_results.zip", recursive=True)
found = glob.glob("/kaggle/input/**/results/benchmark/metrics.jsonl", recursive=True)
if z_final:
    zipfile.ZipFile(z_final[0]).extractall(REPO); print("restored", z_final[0])
elif z_s1:
    zipfile.ZipFile(z_s1[0]).extractall(REPO); print("restored", z_s1[0])
elif found:
    shutil.copytree(os.path.dirname(found[0]), BENCH, dirs_exist_ok=True); print("restored", found[0])
else:
    print("Không có kết quả cũ -> chạy lại Phần A")
    run(f"python scripts/run_benchmark.py --source openml --max_rows 3000 --n_folds 3 "
        f"--out {BENCH} --val_preds --methods logreg,knn,rf,xgb,lgbm,catboost,tabpfn_v2",
        "logs/partA.log")
import json
def merge_into(src, dst):
    """Copy predictions and append new metric rows of src into dst."""
    os.makedirs(f"{dst}/preds", exist_ok=True)
    for f in glob.glob(f"{src}/preds/*.npz"):
        shutil.copy(f, f"{dst}/preds/")
    seen = set()
    if os.path.exists(f"{dst}/metrics.jsonl"):
        for l in open(f"{dst}/metrics.jsonl"):
            r = json.loads(l); seen.add((r["dataset"], r["fold"], r["method"], r.get("status")))
    n = 0
    if os.path.exists(f"{src}/metrics.jsonl"):
        with open(f"{dst}/metrics.jsonl", "a") as out:
            for l in open(f"{src}/metrics.jsonl"):
                r = json.loads(l)
                if (r["dataset"], r["fold"], r["method"], r.get("status")) not in seen:
                    out.write(l if l.endswith("\n") else l + "\n"); n += 1
    if not os.path.exists(f"{dst}/datasets.csv") and os.path.exists(f"{src}/datasets.csv"):
        shutil.copy(f"{src}/datasets.csv", dst)
    print(f"merged {n} rows from {src}")
print(len(glob.glob(f"{BENCH}/preds/*.npz")), "prediction files")

## 2. Chạy OOF-PHE-GBX: dataset chia đôi cho 2 GPU

Mỗi GPU một tiến trình, ghi vào thư mục riêng, sau đó gộp. Cell in dòng log cuối sau mỗi 2 phút.
Nếu session bị ngắt, chạy lại cell: các (dataset, fold) đã xong sẽ được bỏ qua.

In [ ]:
OOF_FOLDS, GBX_INNER, GBX_ITERS, CAT_ITERS = 5, 3, 300, 500
common = (f"--source openml --max_rows 3000 --n_folds 3 --methods oofphe "
          f"--oof_folds {OOF_FOLDS} --gbx_inner {GBX_INNER} --gbx_iterations {GBX_ITERS} "
          f"--cat_iterations {CAT_ITERS}")
run("python scripts/prefetch.py --source openml --max_rows 3000", "logs/prefetch.log")
n_gpu = max(1, torch.cuda.device_count())
shards = [f"results/oof_shard{i}" for i in range(n_gpu)]
jobs = [(f"python scripts/run_benchmark.py {common} --shard {i} --n_shards {n_gpu} --out {d}",
         i, f"logs/oof_shard{i}.log") for i, d in enumerate(shards)]
if n_gpu >= 2:
    run_parallel(jobs)
else:
    run(jobs[0][0], jobs[0][2])
for d in shards:
    merge_into(d, BENCH)
!grep -c "ERR" logs/oof_shard*.log || true

## 3. Báo cáo cuối + đóng gói

Bảng Phần B so sánh Safe OOF-PHE-GBX với TabPFN v2 gốc và mọi phương pháp đã chạy trước đó, kèm
W/T/L và Wilcoxon cho AUC, log-loss, accuracy, mức thua tệ nhất; ablation 5 biến thể; quyết định
của cổng; trọng số thành viên và nhiệt độ.

In [ ]:
!python scripts/make_report.py --bench {BENCH} --out results/report
!cd {REPO} && zip -qr {WORK}/final_results_v2.zip results logs
from IPython.display import Markdown, display, Image
display(Markdown(open("results/report/report.md").read()))
for f in sorted(glob.glob("results/report/*.png")):
    display(Image(f))
print("-> TẢI VỀ:", f"{WORK}/final_results_v2.zip")